## 1. Setup

In [3]:
import logging
from pathlib import Path
from typing import Any, Optional

import joblib
import numpy as np
import pandas as pd
from scipy.stats import entropy

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("xai_engine")

DEFAULT_K_NEIGHBORS = 30
DEFAULT_MAX_CANDIDATES = 4
EPISTEMIC_THRESHOLD_BITS = 0.15

SUPPORTED_MODELS = ("discharge", "transfer", "stay")

MODEL_FILES = {
    "discharge": ("discharge_outcome_model.joblib", "discharge_outcome_metadata.joblib"),
    "transfer": ("transfer_model.joblib", "transfer_metadata.joblib"),
    "stay": ("stay_duration_model.joblib", "stay_duration_metadata.joblib"),
}

DATA_PATH = "Preprocessed_Dataset.csv"
MODEL_DIR = "artifacts"

print("Setup complete.")

Setup complete.


## 2. Errors

Typed exceptions so a bad request surfaces as a clean, specific error — not a raw stack trace —
once this is behind an API.

In [4]:
class XAIEngineError(Exception):
    """Raised for any XAI engine failure that should surface as a clean API error."""


class UnknownModelError(XAIEngineError):
    """Requested model_key is not one of the three supported models."""


class ModelLoadError(XAIEngineError):
    """A required model or metadata file could not be loaded."""


print("Error types defined.")

Error types defined.


## 3. Loading models and training data

Loads all three models and metadata once. Rebuilds `total_days` from the ward-day columns —
identical logic to the Stay Duration and Discharge Outcome notebooks, so the training pool used
for finding "similar patients" matches exactly what those models were trained on.

Every load step validates what it expects to find and raises a specific, readable error if a
file or column is missing, rather than failing deep inside a later calculation.

In [5]:
def load_models(model_dir=MODEL_DIR):
    models = {}
    model_dir = Path(model_dir)
    for key, (model_file, meta_file) in MODEL_FILES.items():
        model_path = model_dir / model_file
        meta_path = model_dir / meta_file
        if not model_path.exists() or not meta_path.exists():
            raise ModelLoadError(
                f"Missing model files for '{key}': expected {model_path} and {meta_path}")
        try:
            model = joblib.load(model_path)
            meta = joblib.load(meta_path)
        except Exception as exc:
            raise ModelLoadError(f"Failed to load model '{key}': {exc}") from exc

        required_meta_keys = {"features", "cat_features", "num_features",
                              "age_order", "time_order", "rare_maps"}
        missing = required_meta_keys - meta.keys()
        if missing:
            raise ModelLoadError(f"Metadata for '{key}' is missing keys: {missing}")

        models[key] = {"model": model, **meta}
        logger.info("Loaded model '%s' (%d features)", key, len(meta["features"]))
    return models


def consolidate_rare(frame, meta):
    out = frame.copy()
    for col, keep in meta.get("rare_maps", {}).items():
        if col in out.columns:
            keep_set = set(keep)
            out[col] = out[col].where(out[col].isna() | out[col].isin(keep_set), "other")
    return out


def load_training_pools(models, data_path=DATA_PATH):
    data_path = Path(data_path)
    if not data_path.exists():
        raise ModelLoadError(f"Training data not found at {data_path}")

    df_raw = pd.read_csv(data_path)
    for c in df_raw.columns:
        if df_raw[c].dtype == object:
            df_raw[c] = (df_raw[c].astype(str).str.strip().str.lower()
                        .replace({"nan": np.nan, "none": np.nan}))
    df_raw = df_raw.rename(columns={"Collision With ": "Collision With"})

    ward_cols = ["Number of days in first ward", "Number of days in Second ward",
                "Number of days in Third ward"]
    missing_ward_cols = [c for c in ward_cols if c not in df_raw.columns]
    if missing_ward_cols:
        raise ModelLoadError(f"Expected ward-day columns not found: {missing_ward_cols}")
    for c in ward_cols:
        df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")
        reason_col = f"{c}_missing_reason"
        if reason_col in df_raw.columns:
            early = df_raw[reason_col] == "early_discharge"
            df_raw.loc[early, c] = df_raw.loc[early, c].fillna(0)
    df_raw["total_days"] = df_raw[ward_cols].sum(axis=1, min_count=1)

    pools = {}
    for key, meta in models.items():
        missing_features = [f for f in meta["features"] if f not in df_raw.columns]
        if missing_features:
            raise ModelLoadError(f"Model '{key}' expects missing features: {missing_features}")
        pool = df_raw[meta["features"]].copy()
        pool = consolidate_rare(pool, meta)
        pools[key] = pool
        logger.info("Loaded training pool for '%s': %d rows", key, len(pool))
    return pools


MODELS = load_models()
TRAIN_POOLS = load_training_pools(MODELS)
print(f"\nReady. Models loaded: {list(MODELS.keys())}")

INFO:xai_engine:Loaded model 'discharge' (18 features)
INFO:xai_engine:Loaded model 'transfer' (21 features)
INFO:xai_engine:Loaded model 'stay' (8 features)
INFO:xai_engine:Loaded training pool for 'discharge': 6178 rows
INFO:xai_engine:Loaded training pool for 'transfer': 6178 rows
INFO:xai_engine:Loaded training pool for 'stay': 6178 rows



Ready. Models loaded: ['discharge', 'transfer', 'stay']


## 4. Internal explanation logic

The engine uses the real model to compare the current record with plausible alternatives from similar training records.
These calculations support the explanation but are not part of the doctor-facing response.

The API exposes only four lines: prediction, model estimate, most influential detail, and most useful missing information.

In [6]:
def encode_ordinal(frame, meta):
    out = frame.copy()
    for col, order in (("Age", meta["age_order"]),
                      ("Time Taken To Reach Hospital", meta["time_order"])):
        if col in out.columns:
            out[col] = out[col].map({v: i for i, v in enumerate(order)})
    return out


def predict_proba_row(model_key, record):
    meta = MODELS[model_key]
    row = pd.DataFrame([{f: record.get(f) for f in meta["features"]}])
    row = encode_ordinal(row, meta)
    try:
        proba = meta["model"].predict_proba(row)[0]
    except Exception as exc:
        raise XAIEngineError(f"Model '{model_key}' failed to score the given record: {exc}") from exc
    classes = list(meta["model"].classes_)
    return proba, classes


def entropy_bits(proba):
    return float(entropy(proba, base=2))


def gower_distance(model_key, record, exclude_field=None):
    meta = MODELS[model_key]
    pool = TRAIN_POOLS[model_key]
    features = [f for f in meta["features"] if f != exclude_field]
    ord_cols = {"Age": meta["age_order"], "Time Taken To Reach Hospital": meta["time_order"]}

    n = len(pool)
    total_dist = np.zeros(n)
    total_weight = np.zeros(n)

    for f in features:
        rv = record.get(f)
        if rv is None or (isinstance(rv, float) and pd.isna(rv)):
            continue
        if f in ord_cols:
            order = ord_cols[f]
            order_map = {v: i for i, v in enumerate(order)}
            if rv not in order_map:
                continue
            rv_idx = order_map[rv]
            pool_idx = pool[f].map(order_map)
            known = pool_idx.notna().values
            d = (pool_idx - rv_idx).abs().values / (len(order) - 1)
            total_dist[known] += np.nan_to_num(d[known])
            total_weight[known] += 1.0
        else:
            pool_vals = pool[f].values
            known = pd.notna(pool_vals)
            d = np.where(pool_vals == rv, 0.0, 1.0)
            total_dist[known] += d[known]
            total_weight[known] += 1.0

    with np.errstate(invalid="ignore", divide="ignore"):
        avg = np.where(total_weight > 0, total_dist / np.maximum(total_weight, 1), np.inf)
    return avg, total_weight


def plausible_values(model_key, record, field, k=DEFAULT_K_NEIGHBORS,
                     max_candidates=DEFAULT_MAX_CANDIDATES, exclude_current=False):
    d, w = gower_distance(model_key, record, exclude_field=field)
    valid = w >= 1
    pool = TRAIN_POOLS[model_key]
    idx_sorted = np.argsort(np.where(valid, d, np.inf))
    neighbor_idx = idx_sorted[:k]
    neighbor_vals = pool[field].iloc[neighbor_idx].dropna()

    if exclude_current:
        current = record.get(field)
        neighbor_vals = neighbor_vals[neighbor_vals != current]

    if len(neighbor_vals) == 0:
        return []

    counts = neighbor_vals.value_counts(normalize=True)
    top = counts.head(max_candidates)
    return list(zip(top.index.tolist(), top.values.tolist()))


# Self-check: distance from a patient to themself must be exactly 0
_sample = TRAIN_POOLS["discharge"].iloc[0].to_dict()
_d, _w = gower_distance("discharge", _sample)
assert _d[0] == 0.0, "self-distance should be 0"
print("Core prediction and distance functions loaded; self-distance check passed.")

Core prediction and distance functions loaded; self-distance check passed.


In [12]:
def simulate_missing_field(model_key, record, field, k=DEFAULT_K_NEIGHBORS,
                           max_candidates=DEFAULT_MAX_CANDIDATES):
    current_proba, _ = predict_proba_row(model_key, record)
    current_entropy = entropy_bits(current_proba)

    candidates = plausible_values(model_key, record, field, k=k, max_candidates=max_candidates)
    if not candidates:
        return None

    sim_entropies, weights = [], []
    for value, weight in candidates:
        modified = dict(record)
        modified[field] = value
        proba, _ = predict_proba_row(model_key, modified)
        sim_entropies.append(entropy_bits(proba))
        weights.append(weight)

    weighted_avg_entropy = float(np.average(sim_entropies, weights=weights))
    return {"field": field, "current_entropy_bits": round(current_entropy, 4),
            "expected_post_entropy_bits": round(weighted_avg_entropy, 4),
            "expected_reduction_bits": round(current_entropy - weighted_avg_entropy, 4),
            "n_candidates_simulated": len(candidates)}


def decision_from_proba(model_key, proba, classes):
    """Apply the deployed model's decision rule, including transfer's threshold."""
    if model_key == "transfer":
        yes_idx = classes.index("yes")
        event_probability = float(proba[yes_idx])
        threshold = float(MODELS[model_key].get("chosen_threshold", 0.5))
        prediction = "yes" if event_probability >= threshold else "no"
        prediction_idx = classes.index(prediction)
        return prediction, event_probability, float(proba[prediction_idx])

    prediction_idx = int(np.argmax(proba))
    prediction = classes[prediction_idx]
    prediction_probability = float(proba[prediction_idx])
    return prediction, prediction_probability, prediction_probability


def simulate_known_field(model_key, record, field, k=DEFAULT_K_NEIGHBORS,
                         max_candidates=DEFAULT_MAX_CANDIDATES):
    current_proba, classes = predict_proba_row(model_key, record)
    pred_class, event_prob, pred_confidence = decision_from_proba(model_key, current_proba, classes)

    candidates = plausible_values(model_key, record, field, k=k, max_candidates=max_candidates,
                                  exclude_current=True)
    if not candidates:
        return None

    results = []
    for value, weight in candidates:
        modified = dict(record)
        modified[field] = value
        proba, _ = predict_proba_row(model_key, modified)
        new_class, _, new_confidence = decision_from_proba(model_key, proba, classes)
        confidence_drop = pred_confidence - new_confidence
        results.append({"alternative_value": value, "resulting_prediction": new_class,
                        "confidence_drop": round(confidence_drop, 4),
                        "confidence_shift_abs": round(abs(confidence_drop), 4),
                        "flips_prediction": new_class != pred_class})

    most_sensitive = max(results, key=lambda r: r["confidence_shift_abs"])

    return {"field": field, "current_prediction": pred_class,
            "current_probability": round(event_prob, 4), "alternatives": results,
            "max_confidence_shift": most_sensitive["confidence_shift_abs"],
            "most_sensitive_alternative": most_sensitive["alternative_value"],
            "any_flip": any(r["flips_prediction"] for r in results)}


print("Simulation functions loaded with model-specific decision rules.")

Simulation functions loaded with model-specific decision rules.


## 5. The `explain` function — doctor-facing payload

This is the single function the API route should call. It returns five plain-language lines:

1. The prediction.
2. The estimated likelihood and a simple confidence label.
3. The most influential recorded detail and a plausible alternative.
4. What the prediction means for this specific model.
5. The most useful missing information, or a data-completeness note.

Technical diagnostics such as entropy, Gower distance, candidate values, confidence shifts, and prediction
flips stay inside the engine and are not shown to the doctor.

In [18]:
def _confidence_label(probability):
    if probability >= 0.75:
        return "high"
    if probability >= 0.50:
        return "moderate"
    return "low"


def _field_label(field):
    labels = {
        "Site of Injury No1": "main injury site",
        "Site of injury No 2": "second injury site",
        "Type of injury No 1": "main injury type",
        "Type of Injury No 2": "second injury type",
        "Mode of Transport to the Hospital": "transport to hospital",
        "Time Taken To Reach Hospital": "time taken to reach hospital",
        "Transfer To Next Hospital": "transfer status",
        "Investigation Done": "investigation performed",
        "Management Name 1": "primary management",
        "Management Name 2": "secondary management",
    }
    return labels.get(field, field.lower())


def _prediction_label(model_key, prediction):
    if model_key == "transfer":
        return f"Prediction: Transfer is {'likely' if prediction == 'yes' else 'unlikely'}."
    if model_key == "stay":
        return f"Prediction: Expected hospital stay is {prediction}."
    return f"Prediction: Expected discharge outcome is {prediction}."


def _interpretation_line(model_key, pred_class):
    if model_key == "transfer":
        threshold = MODELS[model_key].get("chosen_threshold", 0.5)
        return (f"Meaning: Transfer is labelled likely at an estimated transfer probability of "
                f"{float(threshold):.0%} or higher; this is a decision-support flag, not an instruction to transfer.")
    if model_key == "stay":
        return (f"Meaning: This is an estimate of the patient's stay-duration group ({pred_class}), "
                "not a guaranteed discharge date or bed requirement.")
    return (f"Meaning: This is an estimate of the recorded discharge-outcome group ({pred_class}), "
            "not a diagnosis or a replacement for clinical assessment.")


def explain(model_key, record, k=30, max_candidates=4, epistemic_threshold_bits=0.15):
    """Return five descriptive, plain-language lines using the deployed decision rule."""
    if model_key not in SUPPORTED_MODELS:
        raise UnknownModelError(f"'{model_key}' is not supported. Valid: {SUPPORTED_MODELS}")

    meta = MODELS[model_key]
    features = meta["features"]
    missing_fields = [f for f in features
                      if record.get(f) is None or (isinstance(record.get(f), float)
                                                   and pd.isna(record.get(f)))]
    present_count = len(features) - len(missing_fields)
    completeness_floor = float(meta.get("completeness_floor", 0.0))

    if present_count / len(features) < completeness_floor:
        missing_label = ", ".join(_field_label(field) for field in missing_fields[:3])
        return {
            "model": model_key,
            "doctor_explanation": [
                "Prediction: A reliable prediction cannot be made yet.",
                f"Estimated likelihood: Not available; only {present_count} of {len(features)} expected details are recorded.",
                "Why: The model should not be interpreted with this amount of missing information.",
                "Meaning: A model estimate can be provided once enough information is recorded; it is not a diagnosis.",
                f"Next useful detail: record {missing_label}.",
            ],
        }

    proba, classes = predict_proba_row(model_key, record)
    pred_class, pred_prob, confidence_probability = decision_from_proba(model_key, proba, classes)
    confidence = _confidence_label(confidence_probability)

    cf_results = []
    for field in features:
        if field not in missing_fields:
            result = simulate_known_field(model_key, record, field, k=k,
                                          max_candidates=max_candidates)
            if result is not None:
                cf_results.append(result)
    cf_results.sort(key=lambda result: result["max_confidence_shift"], reverse=True)

    if cf_results:
        hinge = cf_results[0]
        field_name = _field_label(hinge["field"])
        current_value = record.get(hinge["field"])
        if hinge["any_flip"]:
            alternative = next(item for item in hinge["alternatives"] if item["flips_prediction"])
            reason_line = (f"Why: The estimate is mainly influenced by {field_name}, recorded as "
                           f"'{current_value}'. A plausible alternative, "
                           f"'{alternative['alternative_value']}', would change the result to "
                           f"{alternative['resulting_prediction']}.")
        else:
            reason_line = (f"Why: The estimate is mainly influenced by {field_name}, recorded as "
                           f"'{current_value}'. Plausible alternatives change the confidence, "
                           f"but not the predicted result.")
    else:
        reason_line = "Why: No single recorded detail clearly determines this estimate; it reflects the combination of available information."

    if missing_fields:
        voi_results = []
        for field in missing_fields:
            result = simulate_missing_field(model_key, record, field, k=k,
                                            max_candidates=max_candidates)
            if result is not None:
                voi_results.append(result)
        voi_results.sort(key=lambda result: result["expected_reduction_bits"], reverse=True)
        if voi_results and voi_results[0]["expected_reduction_bits"] >= epistemic_threshold_bits:
            missing_line = (f"Next useful detail: {_field_label(voi_results[0]['field'])} is not recorded; "
                            f"adding it would most likely clarify this estimate.")
        else:
            missing_line = (f"Data note: {len(missing_fields)} expected detail(s) are not recorded, "
                            "but none is expected to materially resolve the uncertainty.")
    else:
        missing_line = "Data note: All expected details for this model are recorded."

    probability_line = (
        f"Estimated transfer likelihood: {pred_prob:.0%}. This is a {confidence}-confidence model estimate, not a clinical certainty."
        if model_key == "transfer" else
        f"Estimated likelihood: {pred_prob:.0%}. This is a {confidence}-confidence model estimate, not a clinical certainty."
    )

    return {
        "model": model_key,
        "doctor_explanation": [
            _prediction_label(model_key, pred_class),
            probability_line,
            reason_line,
            _interpretation_line(model_key, pred_class),
            missing_line,
        ],
    }


print("Doctor-facing explain() defined with five descriptive lines.")

Doctor-facing explain() defined with five descriptive lines.


## 6. Test before you integrate

Run this section after the model files are loaded. It checks the four-line doctor response, sparse-record refusal,
the transfer threshold rule, and invalid model handling.

In [19]:
import json as _json

print("=== Test 1: sparse record ===")
complete = TRAIN_POOLS["discharge"].dropna()
full_record = complete.iloc[0].to_dict()
sparse_record = {"Age": full_record["Age"], "Site of Injury No1": full_record["Site of Injury No1"]}
result_1 = explain("discharge", sparse_record)
assert len(result_1["doctor_explanation"]) == 5
print("\n".join(result_1["doctor_explanation"]))

print("\n=== Test 2: fully-specified record ===")
result_2 = explain("transfer", TRAIN_POOLS["transfer"].dropna().iloc[0].to_dict())
assert len(result_2["doctor_explanation"]) == 5
print("\n".join(result_2["doctor_explanation"]))

print("\n=== Test 3: decision-rule checks ===")
assert decision_from_proba("transfer", np.array([0.45, 0.55]), ["no", "yes"])[0] == "no"
assert decision_from_proba("transfer", np.array([0.37, 0.63]), ["no", "yes"])[0] == "yes"
assert decision_from_proba(
    "discharge", np.array([0.20, 0.30, 0.50]),
    ["complete recovery", "further interventions", "partial recovery"])[0] == "partial recovery"
assert decision_from_proba(
    "stay", np.array([0.20, 0.30, 0.50]),
    ["0-1 days", "2-3 days", "4+ days"])[0] == "4+ days"
print("Transfer threshold: 55% -> no; 63% -> yes. Multiclass models use highest probability.")

print("\n=== Test 4: bad model_key raises cleanly ===")
try:
    explain("not_a_real_model", {})
    print("FAIL: should have raised UnknownModelError")
except UnknownModelError as exc:
    print(f"Raised correctly: {exc}")

print("\n=== Five-line JSON payload ===")
print(_json.dumps(result_1, indent=2, default=str))

=== Test 1: sparse record ===
Prediction: A reliable prediction cannot be made yet.
Estimated likelihood: Not available; only 2 of 18 expected details are recorded.
Why: The model should not be interpreted with this amount of missing information.
Meaning: A model estimate can be provided once enough information is recorded; it is not a diagnosis.
Next useful detail: record transport to hospital, collision with, approximate speed.

=== Test 2: fully-specified record ===
Prediction: Transfer is likely.
Estimated transfer likelihood: 74%. This is a moderate-confidence model estimate, not a clinical certainty.
Why: The estimate is mainly influenced by time taken to reach hospital, recorded as 'less than 15 minutes'. A plausible alternative, '30 minutes - 1 hour', would change the result to no.
Meaning: Transfer is labelled likely at an estimated transfer probability of 62% or higher; this is a decision-support flag, not an instruction to transfer.
Data note: All expected details for this m